<a href="https://colab.research.google.com/github/kpramod21/AI_UseCases_Projects/blob/main/banking_customer_support_ai_agent_student_workbook_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project: Banking Customer Support AI Agent

## Student Workbook

In this capstone, you will design and build a simple multi-agent customer support assistant for a banking scenario.

The goal is not to write perfect production code. The goal is to practice the AI engineering workflow:

- Understand the problem
- Break the system into components
- Design prompts for each component
- Generate and revise code
- Test the workflow
- Evaluate what worked and what failed

You may use ChatGPT, Gemini, Claude, or another AI assistant to help generate code. Your job is to guide the assistant with clear prompts, test the output, and improve it.

## 1. Understand the Problem

Your assistant should handle three kinds of customer messages:

1. **Positive Feedback**  
   Example: "Thanks for resolving my credit card issue."

2. **Negative Feedback**  
   Example: "My debit card replacement still hasn't arrived."

3. **Ticket Query**  
   Example: "Can you check the status of ticket 650932?"

The system should:

- Classify the message
- Route it to the correct agent
- Generate a response
- Create tickets for negative feedback
- Look up ticket status for queries

### Discussion Questions

Before writing code, answer these:

1. What are the main tasks this system must perform?
2. Which tasks require AI?
3. Which tasks can be solved with regular Python code?
4. Why might multiple agents be useful here?

In [ ]:
# Write brief answers as comments.
# 1. Figure out what policies are there and how they work. From there, put the policies into a RAG for the system to query.
# 2. CLASSIFIER: Message classification: classify incoming messages as positive, negative, query. (AI yes! User sentiment)
# 3. Router -- route messages to the appropriate agent based on the classification.  (AI no! Rule-based) if negative -> ticketing agent, if query -> query response agent, if positive -> thank you agent.
# 4. Ticketing agent -- create a ticket for negative messages, and generate appropriate response.  (AI yes, for response generation, but ticket creation can be rule-based)
# 5. Query response agent -- look up ticket status for query messages. (Ticket status lookup is rule based, but response generation can be AI, or not.)
# 6. Thank you agent -- send thank you messages for positive feedback. (AI or rule-based depending on how personalized we want the responses to be.)
# 7?. (allow retry in case of failure -- flag for follow-up.)
# 8?. Logging agent -- log all interactions for future reference and analysis.


# How do we handle FAQs?
# -> 1. From user question, compute similarity with existing FAQs using a vector database (e.g., FAISS), and return most relevant FAQ (if similarity above threshold) as response.
# -> 2. From user question, build a RAG with policy and other relevant information, and use that to generate a response.
# -> In both cases, we'd need some type of judge agent to OK the answer.

## 2. Proposed Architecture

```text
User Message
      |
      v
Classifier Agent
      |
      +-----------------------+
      |                       |
      v                       v
Feedback Agent           Query Agent
      |                       |
      +-----------+-----------+
                  |
                  v
             Ticket Store
```

### Agent Responsibilities

| Agent | Responsibility |
|---|---|
| Classifier Agent | Decide whether the message is positive feedback, negative feedback, or a query |
| Feedback Agent | Respond to feedback and create new tickets for negative feedback |
| Query Agent | Extract a ticket number and return its status |
| Orchestrator | Connect the agents and route the message |

### Architecture Reflection

1. What is the responsibility of each agent?
2. What happens if the classifier makes a mistake?
3. Could this be built as one big prompt? What would be the downside?

In [ ]:
# Architecture notes

## 3. Setup

Keep setup simple. You can use only Python at first. Add an LLM only where it helps.

Suggested imports:

In [1]:
import re
import random
import json
from pathlib import Path

## 4. Build the Classifier Agent

### Goal

Classify a customer message into exactly one of these categories:

- `Positive Feedback`
- `Negative Feedback`
- `Query`

You may use:

- rule-based logic
- an LLM prompt
- a hybrid approach

### Starter Prompt

```text
Create a Python ClassifierAgent.

Classify customer messages into:
- Positive Feedback
- Negative Feedback
- Query

Return only the category.
```

### What's wrong with this prompt?

Think about:

- Does it include examples?
- Does it define ambiguous cases?
- Does it specify exact output values?
- Does it explain what to do with mixed messages?

In [ ]:
class ClassifierAgent_old:
    def classify(self, message: str) -> str:
        text = message.lower()

        if any(word in text for word in ["thanks", "thank you", "appreciate", "great service"]):
            return "Positive Feedback"

        if any(word in text for word in ["status", "ticket", "update", "check"]):
            return "Query"

        if any(word in text for word in ["issue", "problem", "not working", "unresolved", "still waiting", "hasn't arrived"]):
            return "Negative Feedback"


        return "Query"

In [ ]:
# Notes: what is wrong with the starter prompt?

### Better Prompt

Write a short improved prompt. Do not make it long. Make it clear.

In [ ]:
better_classifier_prompt = """
Create a Python ClassifierAgent.

Classify customer messages into exactly one category:
- Positive Feedback
- Negative Feedback
- Query

Return only the category name.
Use OpenAI API to classify the message based on keywords and context.

Examples:
"Thanks for your help" -> Positive Feedback
"My issue is still unresolved" -> Negative Feedback
"What is the status of ticket 123456?" -> Query
"""

In [12]:
from openai import OpenAI
#from dotenv import load_dotenv
#load_dotenv()

from google.colab import userdata

# Removed: import openai (as it conflicts with the new API style where OpenAI is a client)

class ClassifierAgent:
    """
    Classifies customer messages into one of the following categories:
    - Positive Feedback
    - Negative Feedback
    - Query
    """

    def __init__(self, model="gpt-3.5-turbo"):
        self.model = model
        # Retrieve the API key from Colab secrets and pass it to the OpenAI client
        api_key = userdata.get('OPENAI_API_KEY')
        self.client = OpenAI(api_key=api_key) # Initialize the OpenAI client with the API key

    def classify(self, message: str) -> str:
        """
        Classify the customer message.

        Parameters:
            message (str): Customer input message.

        Returns:
            str: One of:
                 - Positive Feedback
                 - Negative Feedback
                 - Query
        """

        prompt = f"""
You are a banking customer support message classifier.

Classify the customer message into exactly one category:
- Positive Feedback
- Negative Feedback
- Query

Definitions:
- Positive Feedback: Appreciation, gratitude, praise, or satisfaction.
- Negative Feedback: Complaints, dissatisfaction, unresolved issues, frustration, or service problems.
- Query: Requests for information, ticket status checks, or questions.

Return ONLY the category name.
Do not provide explanations.

Examples:
"Thanks for your help" -> Positive Feedback
"My issue is still unresolved" -> Negative Feedback
"What is the status of ticket 123456?" -> Query

Customer Message:
"{message}"
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        category = response.choices[0].message.content.strip()

        valid_categories = [
            "Positive Feedback",
            "Negative Feedback",
            "Query"
        ]

        if category not in valid_categories:
            return "Negative Feedback"

        return category

### Generate or Write Your Classifier Code

Use your improved prompt with an AI assistant, then paste or write the generated code below.

In [ ]:
class ClassifierAgent_rev2:
    POSITIVE_KEYWORDS = {
        "thanks", "thank you", "appreciate", "great", "excellent",
        "helpful", "resolved", "satisfied", "happy"
    }

    NEGATIVE_KEYWORDS = {
        "issue", "problem", "unresolved", "complaint", "waiting",
        "delayed", "not working", "failed", "frustrated", "arrived"
    }

    QUERY_KEYWORDS = {
        "status", "ticket", "check", "update", "query", "when",
        "where", "could you", "can you"
    }

    def classify(self, message: str) -> str:
        text = message.lower()

        # Queries take priority because they often contain ticket numbers.
        if any(keyword in text for keyword in self.QUERY_KEYWORDS):
            return "Query"

        # Negative feedback takes priority over positive in mixed messages.
        if any(keyword in text for keyword in self.NEGATIVE_KEYWORDS):
            return "Negative Feedback"

        if any(keyword in text for keyword in self.POSITIVE_KEYWORDS):
            return "Positive Feedback"

        # Default fallback
        return "Query"

## 5. Test the Classifier

Try the provided test cases and add your own.

In [13]:
classifier = ClassifierAgent()

test_messages = [
    "Thanks for your help",
    "Thanks for your help, but my issue is still unresolved",
    "What is the status of ticket 123456?",
    "Thanks, but my issue still is not fixed.",
    "Yeah, this would be great support if I lived in opposite-land. You guys really stink!"
]

for msg in test_messages:
    print(msg, "->", classifier.classify(msg))

Thanks for your help -> Positive Feedback
Thanks for your help, but my issue is still unresolved -> Negative Feedback
What is the status of ticket 123456? -> Query
Thanks, but my issue still is not fixed. -> Negative Feedback
Yeah, this would be great support if I lived in opposite-land. You guys really stink! -> Negative Feedback


### Classifier Test Notes

Record what worked and what failed.

| Message | Expected | Actual | Notes |
|---|---|---|---|
| Thanks for your help | Positive Feedback | | |
| My issue is still unresolved | Negative Feedback | | |
| What is the status of ticket 123456? | Query | | |
| Thanks, but my issue still is not fixed. | Discuss | | |

## 6. Build the Feedback Agent

### Goal

The Feedback Agent handles both positive and negative feedback.

For positive feedback:

- Generate a brief professional thank-you message

For negative feedback:

- Generate a unique 6-digit ticket number
- Save the ticket with status `Unresolved`
- Return an empathetic response with the ticket number

### Starter Prompt

```text
Create a FeedbackAgent.

Generate thank-you messages for positive feedback and apology messages for negative feedback.
```

### What's wrong with this prompt?

Think about:

- Does it specify ticket creation?
- Does it specify ticket format?
- Does it define where tickets are stored?
- Does it limit response length or tone?

In [ ]:
from openai import OpenAI

client = OpenAI()


class FeedbackAgent:
    def handle_feedback(
        self,
        feedback_type: str,
        customer_name: str = "Customer"
    ) -> str:

        if feedback_type == "Positive Feedback":
            prompt = f"""
Generate a warm and professional thank-you message for a banking customer.

Customer Name: {customer_name}

Keep the response to 1-2 sentences.
"""

        elif feedback_type == "Negative Feedback":
            prompt = f"""
Generate a polite and empathetic apology message for a banking customer.

Customer Name: {customer_name}

A support ticket has been created.

Keep the response to 1-2 sentences.
"""

        else:
            return "Invalid feedback type."

        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt,
            temperature=0.7,
        )

        return response.output_text.strip()

In [ ]:
# Notes: what is wrong with the starter prompt?
# Too generic - not enough info to create a personalized response. We can improve it by adding customer name and context about the feedback.

### Better Prompt

Here is a better but still short version.

In [14]:
better_feedback_prompt = """
Create a Python FeedbackAgent.

For Positive Feedback:
- Return a warm professional thank-you response
- Keep it to 1-2 sentences
- Do not invent account details

For Negative Feedback:
- Generate a unique 6-digit ticket number
- Save the ticket with status "Unresolved"
- Return an empathetic response that includes the ticket number

Use information from the message to personalize the response, but do not invent any account details or specific information that is not provided.
Use OpenAI API to generate the responses based on the feedback type and customer name.

In responses should be warm and professional.
"""

### Ticket Store

For this capstone, you can use a simple file or dictionary as the ticket store.

A real system would use a database. For learning, simple storage is enough.

In [15]:
TICKET_FILE = Path("tickets.json")

def load_tickets() -> dict:
    if not TICKET_FILE.exists():
        return {}
    return json.loads(TICKET_FILE.read_text())

def save_tickets(tickets: dict) -> None:
    TICKET_FILE.write_text(json.dumps(tickets, indent=2))

# Seed a sample ticket for testing
sample_tickets = load_tickets()
sample_tickets.setdefault("123456", {"status": "Resolved", "message": "Sample ticket"})
save_tickets(sample_tickets)

### Generate or Write Your Feedback Agent Code

In [19]:
from openai import OpenAI
import random
import json
from pathlib import Path
from google.colab import userdata

# Initialize the OpenAI client with the API key from userdata
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))


class FeedbackAgent:
    def __init__(self, ticket_file: str = "tickets.json"):
        self.ticket_file = Path(ticket_file)
        self._initialize_ticket_file()

    def _initialize_ticket_file(self):
        if not self.ticket_file.exists():
            self.ticket_file.write_text("[]")

    def _load_tickets(self) -> list[dict]:
        return json.loads(self.ticket_file.read_text())

    def _save_tickets(self, tickets: list[dict]) -> None:
        self.ticket_file.write_text(json.dumps(tickets, indent=2))

    def _generate_unique_ticket_number(self) -> str:
        tickets = self._load_tickets()

        existing_ticket_numbers = {
            ticket["ticket_number"]
            for ticket in tickets
            if isinstance(ticket, dict) and "ticket_number" in ticket
        }

        while True:
            ticket_number = str(random.randint(100000, 999999))
            if ticket_number not in existing_ticket_numbers:
                return ticket_number

    def _create_ticket(self, customer_name: str, message: str) -> str:
        ticket_number = self._generate_unique_ticket_number()

        tickets = self._load_tickets()
        tickets.append({
            "ticket_number": ticket_number,
            "customer_name": customer_name,
            "message": message,
            "status": "Unresolved"
        })
        self._save_tickets(tickets)

        return ticket_number

    def _generate_response(
        self,
        feedback_type: str,
        customer_name: str,
        message: str,
        ticket_number: str | None = None
    ) -> str:
        prompt = f"""
You are a banking customer support assistant.

Generate a response for this customer.

Feedback Type: {feedback_type}
Customer Name: {customer_name}
Customer Message: {message}
Ticket Number: {ticket_number if ticket_number else "None"}

Requirements:
- Generate a warm and professional response
- Keep the response to 1-2 sentences.
- Do not invent account details.
- If a ticket number is provided, include it.
"""

        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt,
            temperature=0.7,
        )

        return response.output_text.strip()

    def handle_feedback(
        self,
        feedback_type: str,
        message: str,
        customer_name: str = "Customer"
    ) -> str:
        if feedback_type == "Positive Feedback":
            return self._generate_response(
                feedback_type=feedback_type,
                customer_name=customer_name,
                message=message
            )

        if feedback_type == "Negative Feedback":
            ticket_number = self._create_ticket(
                customer_name=customer_name,
                message=message
            )

            return self._generate_response(
                feedback_type=feedback_type,
                customer_name=customer_name,
                message=message,
                ticket_number=ticket_number
            )

        return "Invalid feedback type."

## 7. Test the Feedback Agent

In [20]:
from pathlib import Path

Path("tickets.json").write_text("[]")

classifier = ClassifierAgent()
feedback_agent = FeedbackAgent()

test_messages = [
    "Thanks for your help",
    "Thanks for your help, but my issue is still unresolved",
    "What is the status of ticket 123456?",
    "Thanks, but my issue still is not fixed.",
    "Yeah, this would be great support if I lived in opposite-land. You guys really stink!"
]

for msg in test_messages:
    feedback_type = classifier.classify(msg)
    print(msg)
    print(feedback_agent.handle_feedback(feedback_type, msg))
    print("\n\n")
#print(load_tickets())

Thanks for your help
Dear Customer, 

Thank you for your kind words! We're always here to help whenever you need us.



Thanks for your help, but my issue is still unresolved
Dear Customer, I’m sorry to hear your issue is still unresolved. Please rest assured, we are prioritizing ticket number 766792 and will work diligently to resolve it as quickly as possible.



What is the status of ticket 123456?
Invalid feedback type.



Thanks, but my issue still is not fixed.
Dear Customer, thank you for your patience. We apologize that your issue is still unresolved and will prioritize ticket number 367311 to ensure it is addressed promptly.



Yeah, this would be great support if I lived in opposite-land. You guys really stink!
Dear Customer, we’re truly sorry to hear about your frustration and appreciate your feedback. Please rest assured that we are committed to improving your experience and will look into your concerns promptly under ticket number 103103.





### Feedback Agent Test Notes

1. Did positive feedback return a useful response?
2. Did negative feedback create a ticket?
3. Was the ticket saved correctly?
4. Was the response empathetic and clear?

In [ ]:
# Feedback test notes

## 8. Build the Query Agent

### Goal

The Query Agent should:

- Extract a 6-digit ticket number from the customer message
- Look up the ticket
- Return the ticket status
- Handle missing or invalid ticket numbers

### Starter Prompt

```text
Create a QueryAgent that retrieves ticket status information.
```

### What's wrong with this prompt?

Think about:

- Does it mention ticket number extraction?
- Does it specify ticket format?
- Does it explain what to do when a ticket is missing?
- Does it define the response format?

In [ ]:
import json
import re
from pathlib import Path


class QueryAgent:
    def __init__(self, ticket_file: str = "tickets.json"):
        self.ticket_file = Path(ticket_file)

    def _load_tickets(self) -> list[dict]:
        if not self.ticket_file.exists():
            return []

        data = json.loads(self.ticket_file.read_text())

        # Handle expected format: [{...}, {...}]
        if isinstance(data, list):
            return data

        # Handle alternate format: {"tickets": [{...}, {...}]}
        if isinstance(data, dict) and "tickets" in data:
            return data["tickets"]

        return []

    def _extract_ticket_number(self, message: str) -> str | None:
        match = re.search(r"\b\d{6}\b", message)
        if match:
            return match.group(0)
        return None

    def get_ticket_status(self, message: str) -> str:
        ticket_number = self._extract_ticket_number(message)

        if ticket_number is None:
            return "Please provide a valid 6-digit ticket number."

        tickets = self._load_tickets()

        for ticket in tickets:
            if str(ticket.get("ticket_number")) == ticket_number:
                status = ticket.get("status", "Unknown")
                return f"Your ticket #{ticket_number} is currently marked as: {status}."

        return f"Ticket #{ticket_number} could not be found."

In [ ]:
# Notes: what is wrong with the starter prompt?

### Better Prompt

In [ ]:
better_query_prompt = """
Create a Python QueryAgent.

Requirements:
- Extract a 6-digit ticket number from the customer message
- Look up the ticket in the ticket store
- If found, return: "Your ticket #[TicketNumber] is currently marked as: [Status]."
- If not found, return: "Ticket #[TicketNumber] could not be found."
- If no ticket number is provided, ask the customer to provide a ticket number
"""

In [21]:
import json
import re
import os


class QueryAgent:
    """
    Handles customer ticket status queries.

    Responsibilities:
    - Extract a 6-digit ticket number from the customer message.
    - Look up the ticket in tickets.json.
    - Return the current ticket status if found.
    - Inform the customer if the ticket cannot be found.
    - Ask for a ticket number if none is provided.
    """

    def __init__(self, tickets_file="tickets.json"):
        self.tickets_file = tickets_file

        # Create an empty tickets file if it doesn't exist
        if not os.path.exists(self.tickets_file):
            with open(self.tickets_file, "w") as file:
                json.dump([], file, indent=4)

    def extract_ticket_number(self, message):
        """
        Extract a 6-digit ticket number from the customer message.

        Returns:
            str: Ticket number if found, otherwise None.
        """
        match = re.search(r"\b(\d{6})\b", message)

        if match:
            return match.group(1)

        return None

    def get_ticket_status(self, ticket_number):
        """
        Retrieve the status of the specified ticket.

        Returns:
            str: Ticket status if found, otherwise None.
        """
        try:
            with open(self.tickets_file, "r") as file:
                tickets = json.load(file)

            for ticket in tickets:
                if str(ticket.get("ticket_id")) == ticket_number:
                    return ticket.get("status", "Unknown")

        except (FileNotFoundError, json.JSONDecodeError):
            return None

        return None

    def handle_query(self, message):
        """
        Process the customer query.

        Returns:
            str: Appropriate response based on the query result.
        """
        ticket_number = self.extract_ticket_number(message)

        if not ticket_number:
            return (
                "Could you please provide your 6-digit ticket number "
                "so that I can check the status for you?"
            )

        status = self.get_ticket_status(ticket_number)

        if status:
            return (
                f"Your ticket #{ticket_number} is currently "
                f"marked as: {status}."
            )

        return f"Ticket #{ticket_number} could not be found."

### Generate or Write Your Query Agent Code

In [ ]:
# Query Agent

class _QueryAgent:
    def extract_ticket_number(self, message: str):
        # TODO: extract a 6-digit ticket number using regex
        pass

    def handle_query(self, message: str) -> str:
        # TODO: look up ticket status and return response
        pass

## 9. Test the Query Agent

In [23]:
query_agent = QueryAgent()
# These tests do not cover the case where something was mis-routed.
query_tests = [
    "What is the status of ticket 123456?",
    "Can you check ticket #999999?",
    "Can you check my ticket?",
]

for msg in query_tests:
    print(msg)
    print(query_agent.handle_query(msg))
    print()

What is the status of ticket 123456?
Ticket #123456 could not be found.

Can you check ticket #999999?
Ticket #999999 could not be found.

Can you check my ticket?
Could you please provide your 6-digit ticket number so that I can check the status for you?



### Query Agent Test Notes

| Message | Expected Behavior | Actual Behavior | Notes |
|---|---|---|---|
| What is the status of ticket 123456? | Return status | | |
| Can you check ticket #999999? | Ticket not found | | |
| Can you check my ticket? | Ask for ticket number | | |

## 10. Build the Orchestrator

### Goal

The orchestrator connects the agents.

It should:

1. Receive a customer message
2. Classify the message
3. Route the message to the correct agent
4. Return the final response

### Starter Prompt

```text
Create a controller that connects all agents.
```

### What's wrong with this prompt?

Think about:

- Does it define the workflow?
- Does it specify exact routes?
- Does it explain what each agent returns?
- Does it handle unknown outputs?

In [ ]:
# Notes: what is wrong with the starter prompt?

### Better Prompt

In [ ]:
better_orchestrator_prompt = """
Create a Python OrchestratorAgent.


Workflow:
1. Receive customer message
2. Call ClassifierAgent.classify(message)
3. If category is Positive Feedback or Negative Feedback, call FeedbackAgent.handle_feedback(category,message)
4. If category is Query, call QueryAgent.handle_query(message)
5. Return the final response

Use clear class interfaces and handle unexpected classifier output gracefully.
"""

### Generate or Write Your Orchestrator Code

In [24]:
class OrchestratorAgent:
    def __init__(self, classifier, feedback_agent, query_agent):
        self.classifier = classifier
        self.feedback_agent = feedback_agent
        self.query_agent = query_agent

    def handle_message(self, message: str) -> str:
        category = self.classifier.classify(message)

        if category in ["Positive Feedback", "Negative Feedback"]:
            return self.feedback_agent.handle_feedback(category, message)

        if category == "Query":
            return self.query_agent.handle_query(message)

        return (
            "I'm sorry, I could not determine how to handle your request. "
            "Please rephrase your message."
        )

In [ ]:
# Orchestrator

class _OrchestratorAgent:
    def __init__(self, classifier, feedback_agent, query_agent):
        self.classifier = classifier
        self.feedback_agent = feedback_agent
        self.query_agent = query_agent

    def handle_message(self, message: str) -> str:
        # TODO: classify and route to the correct agent
        pass

## 11. End-to-End Testing

Now test the whole workflow.

In [27]:
orchestrator = OrchestratorAgent(
    classifier=ClassifierAgent(),
    feedback_agent=FeedbackAgent(),
    query_agent=QueryAgent(),
)

end_to_end_tests = [
    "Thanks for resolving my issue.",
    "My debit card still hasn't arrived.",
    "What is the status of ticket 123456?",
    "Thanks, but my issue still is not fixed.",
]

for msg in end_to_end_tests:
    print("User:", msg)
    print("Assistant:", orchestrator.handle_message(msg))
    print("-" * 60)

User: Thanks for resolving my issue.
Assistant: Dear Customer, thank you for your kind words! We’re glad we could resolve your issue and are always here to assist you.
------------------------------------------------------------
User: My debit card still hasn't arrived.
Assistant: Dear Customer, we apologize for the delay in receiving your debit card and are looking into this issue with priority; please refer to ticket number 332751 for updates. Thank you for your patience and understanding.
------------------------------------------------------------
User: What is the status of ticket 123456?
Assistant: Ticket #123456 could not be found.
------------------------------------------------------------
User: Thanks, but my issue still is not fixed.
Assistant: Dear Customer, thank you for your patience. We apologize that your issue is still unresolved and will prioritize ticket number 227595 to ensure it is addressed promptly.
------------------------------------------------------------


### End-to-End Test Notes

| Input | Expected Route | Actual Route | Response Quality Notes |
|---|---|---|---|
| Thanks for resolving my issue. | Positive Feedback | | |
| My debit card still hasn't arrived. | Negative Feedback | | |
| What is the status of ticket 123456? | Query | | |
| Thanks, but my issue still is not fixed. | Discussion | | |

## 12. Evaluation

A working demo is not enough. You should also evaluate your system.

### Classification Evaluation

- Did the classifier choose the correct category?
- Which examples failed?
- Were ambiguous messages handled reasonably?

### Routing Evaluation

- Did the orchestrator send each message to the correct agent?

### Ticket Evaluation

- Were tickets created correctly?
- Were duplicate or missing tickets handled?

### Response Quality Evaluation

- Were responses clear?
- Were responses professional?
- Were negative feedback responses empathetic?

In [31]:
# Create your own evaluation test cases here.
# Add at least 5 more examples.

orchestrator = OrchestratorAgent(
    classifier=ClassifierAgent(),
    feedback_agent=FeedbackAgent(),
    query_agent=QueryAgent(),
)
evaluation_cases = [
    # {"message": "...", "expected_category": "..."},
    "Thank you for helping me regain access to my net banking account.",

    "I reported this issue last week, but my credit card replacement still hasn't arrived.",

    "Could you please check the status of ticket 650932?",

    "The mobile banking app keeps crashing whenever I try to transfer money.",

    "Can you tell me whether ticket 784521 has been resolved yet?"
]

for msg in evaluation_cases:
    print("User:", msg)
    print("Assistant:", orchestrator.handle_message(msg))
    print("Classifier:", classifier.classify(msg))
    print("-" * 60)

User: Thank you for helping me regain access to my net banking account.
Assistant: Dear Customer, thank you for your kind words! We're delighted to have helped you regain access to your net banking account and are always here to assist you.
Classifier: Positive Feedback
------------------------------------------------------------
User: I reported this issue last week, but my credit card replacement still hasn't arrived.
Assistant: Dear Customer, we sincerely apologize for the delay in your credit card replacement; your ticket number 272741 has been prioritized, and we are working to resolve this promptly. Thank you for your patience and understanding.
Classifier: Negative Feedback
------------------------------------------------------------
User: Could you please check the status of ticket 650932?
Assistant: Ticket #650932 could not be found.
Classifier: Query
------------------------------------------------------------
User: The mobile banking app keeps crashing whenever I try to tran

## 13. Stretch Goals

If you finish early, choose one improvement:

- Add a Streamlit UI
- Use SQLite instead of JSON file storage
- Add logs showing classifier output and route taken
- Add a confidence score to classification
- Build a LangGraph version
- Build a CrewAI version
- Add a new specialized agent, such as Credit Card Agent or Loan Agent

## Final Reflection

Answer these questions:

1. Which component was easiest to build?
2. Which component was hardest to build?
3. Which prompt improved the most after revision?
4. Which part did not need AI?
5. What would you improve in a real banking production system?

### Key Takeaway

Good AI systems are not just prompts. They are well-designed workflows with clear responsibilities, reliable non-AI components, testing, and evaluation.

In [ ]:
# Final reflection notes